In [ ]:
import matplotlib.pyplot as plt
import torch
import pandas as pd
import sys
import os
import datasets
import networkx as nx
from matplotlib.patches import Patch
import numpy as np
from datasets import load_from_disk
from itertools import combinations

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
# Add to sys.path only if not already there
if root_path not in sys.path:
    sys.path.append(root_path)
    print(f"Added {root_path} to sys.path")
else:
    print(f"{root_path} is already in sys.path")

colors = plt.cm.Dark2.colors
linewidth = 4
plt.rcParams.update(
    {
        'font.size': 18,
        'text.usetex': True,
        'axes.linewidth': linewidth,
        'xtick.major.width': linewidth,
        'ytick.major.width': linewidth,
        'xtick.major.size': 2*linewidth,
        'ytick.major.size': 2*linewidth,
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",  # STIX fonts are designed to match Times
    }
)

Number of NaN in nash_500 dataset, train split,  coherence attribute:  10300
Number of NaN in nash_500 dataset, train split,  complexity attribute:  9373
Number of NaN in nash_500 dataset, train split,  correctness attribute:  9398
Number of NaN in nash_500 dataset, train split,  helpfulness attribute:  9371
Number of NaN in nash_500 dataset, train split,  verbosity attribute:  9383
Number of NaN in nash_500 dataset, validation split,  coherence attribute:  2543
Number of NaN in nash_500 dataset, validation split,  complexity attribute:  2241
Number of NaN in nash_500 dataset, validation split,  correctness attribute:  2266
Number of NaN in nash_500 dataset, validation split,  helpfulness attribute:  2253
Number of NaN in nash_500 dataset, validation split,  verbosity attribute:  2258

# Evaluation

In [ ]:
# Tuple: (model_name, base_model, lr, kl beta, model specific or None, ckpt)
# Setup your paths to the desired evaluation rewards
data_paths = {
    ("Baseline", "Qwen2.5-0.5B", None, None, None, None): "path/to/baseline__rewards",
    ("RLOO", "Qwen2.5-0.5B", 1e-5, 0.001, None, 1000): "path/to/rloo__rewards",
    ("Nash", "Qwen2.5-0.5B", 1e-5, 0.001, 0.75, 1000): "path/to/nash__rewards",
    ("Stackelberg", "Qwen2.5-0.5B", 1e-5, 0.001, 5.0, 1000): "path/to/stackelberg__rewards",
}

print("Input paths:")
print(data_paths)

df_mean = []
df_std = []
rewards_dict = {}
for name, data_path in data_paths.items():
    try:
        rewards = load_from_disk(data_path)
    except Exception as e:
        print("Loading failed for : ", data_path)
        print("Error: ", e)
        continue
    rewards_dict[name] = rewards
    df_rewards_mean = []
    df_rewards_std = []
    for datasplit_name, datasplit in rewards.items():
        for col_name in datasplit.column_names:
            if col_name == "prompt_id":
                continue
            avg_reward_per_prompt = torch.tensor(datasplit[col_name])
            num_nan = torch.isnan(avg_reward_per_prompt).sum().item()
            if num_nan > 0:
                print(f"Number of NaN in {name} dataset, {datasplit_name} split,  {col_name} attribute: {num_nan} out of {len(datasplit)}")
            df_rewards_mean.append(
                (datasplit_name, col_name, avg_reward_per_prompt.nanmean().item())
            )
            df_rewards_std.append(
                (datasplit_name, col_name, avg_reward_per_prompt.max().item() - avg_reward_per_prompt.min().item())
            )
    df_mean.append(pd.DataFrame(df_rewards_mean, columns=["dataset", "attribute", name]).set_index(["dataset", "attribute"]))
    df_std.append(pd.DataFrame(df_rewards_std, columns=["dataset", "attribute", name]).set_index(["dataset", "attribute"]))
for df_name, df in {"mean": df_mean, "std": df_std}.items():
    df = pd.concat(df, axis=1).T
    df[("train", "mean")] = df["train"].mean(1)
    df[("validation", "mean")] = df["validation"].mean(1)
    print("Dataframe ", df_name)
    with pd.option_context('display.precision', 3):
        display(df.sort_values(("train", "mean"), ascending=False))

# Round Robin Tournament

In [ ]:
models_to_compare = {
    "Baseline0.5": (("Baseline", "Qwen2.5-0.5B", None, None, None, None), 0),
    "RLOO": (("RLOO", "Qwen2.5-0.5B", 1e-5, 0.001, None, 1000), 0),
    "Nash-MD": (("Nash", "Qwen2.5-0.5B", 1e-5, 0.001, 0.75, 1000), 0),
    "StackelbergGDA-Leader": (("Stackelberg", "Qwen2.5-0.5B", 1e-5, 0.001, 5.0, 1000), 0),
    "StackelbergGDA-Follower": (("Stackelberg", "Qwen2.5-0.5B", 1e-5, 0.001, 5.0, 1000), 1),
}
n = len(models_to_compare)

for datasplit in ["train", "validation"]:
    col_names = list(models_to_compare.keys())
    rewards = []
    for col_name, (key, idx) in models_to_compare.items():
        data = rewards_dict[key][datasplit]
        attribute_rewards = []
        for attribute_name in data.column_names:
            if attribute_name == "prompt_id":
                continue
            attribute_rewards.append(torch.tensor(data[attribute_name])[:, idx])  # Shape: (n_prompts, )
        attribute_rewards = torch.stack(attribute_rewards)  # Shape: (n_attributes, n_prompts)
        rewards.append(attribute_rewards)
    rewards = torch.stack(rewards)  # Shape: (n_model, n_attributes, n_prompts)
    comparisons = torch.zeros((n,n))
    for i, j in combinations(range(n), 2):
        arr1 = rewards[i]
        arr2 = rewards[j]
        comp = torch.mean((arr1 >= arr2).float())
        comparisons[i, j] = comp
        comparisons[j, i] = 1-comp
    print(datasplit)
    comparisons = pd.DataFrame(
        comparisons,
        index=col_names,
        columns=col_names,
    )
    with pd.option_context('display.precision', 4):
        display(comparisons)

# Follower Weight ablation

In [ ]:
baseline_model = ("Baseline", "Qwen2.5-0.5B", None, None, None, None)
model_names = [("Stackelberg", "Qwen2.5-0.5B", 1e-5, 0.001, w, 1000) for w in [1.0, 5.0, 10.0]]
winrates = []
for name in model_names:
    rewards = rewards_dict[name]
    print("Calculating: ", name)
    for datasplit_name, datasplit in rewards.items():
        datasplit_comparisons = []
        for col_name in datasplit.column_names:
            if col_name == "prompt_id":
                continue
            avg_reward_per_prompt = torch.tensor(datasplit[col_name])  # Shape: (n_prompts, n_completions)
            avg_reward_per_prompt_baseline = torch.tensor(rewards_dict[baseline_model][datasplit_name][col_name])  # Shape: (n_prompts, n_completions)
            datasplit_comparisons.append((avg_reward_per_prompt >= avg_reward_per_prompt_baseline).float())  # Shape: (n_prompts, n_completions)
        datasplit_comparisons = torch.stack(datasplit_comparisons, 0).mean(0)  # Shape: (n_prompts, n_completions)
        winrates.append(name+(datasplit_name, "Leader", torch.mean(datasplit_comparisons[:,0]).item()))
        winrates.append(name+(datasplit_name, "Follower", torch.mean(datasplit_comparisons[:,1]).item()))
with pd.option_context('display.precision', 3):
    display(
        pd.DataFrame(
            winrates,
            columns=["model", "base_model", "lr", "kl_beta", "model_param", "ckpt", "dataset", "winrate_calculation", "winrate"]
        ).set_index(
            ["model", "base_model", "lr", "kl_beta", "model_param", "ckpt", "winrate_calculation", "dataset"]
        ).unstack(["winrate_calculation", "dataset"]).sort_values(("winrate", "Leader", "train"),ascending=False)
    )

# Inference-time improvement

In [ ]:
# Max@k winrates
models_to_compare = {
    "Qwen2.5-0.5B": ("Baseline", "Qwen2.5-0.5B", None, None, None, None),
    "RLOO": ("RLOO", "Qwen2.5-0.5B", 1e-5, 0.001, None, 1000),
    "Nash-MD": ("Nash", "Qwen2.5-0.5B", 1e-5, 0.001, 0.75, 1000),
    "StackelbergGDA": ("Stackelberg", "Qwen2.5-0.5B", 1e-5, 0.001, 5.0, 1000),
}
baseline_model = "Baseline0.5"
n = len(models_to_compare)
attribute_names = ['coherence', 'complexity', 'correctness', 'helpfulness', 'verbosity']

for datasplit in ["train", "validation"]:
    print(f"--- DATASPLIT: {datasplit} ---")
    rewards_cummax = []
    for attribute_name in attribute_names:
        if attribute_name == "prompt_id":
            continue
        attribute_rewards_cummax = []
        plt.figure(figsize=(10, 6))
        for i, (model_name, model_key) in enumerate(models_to_compare.items()):
            data = rewards_dict[model_key][datasplit][attribute_name]
            arr = torch.tensor(data)  # Shape: (n_prompts, n_completions)
            arr_cummax = torch.cummax(arr, 1).values  # Shape: (n_prompts, n_completions)
            attribute_rewards_cummax.append(arr_cummax)
            plt.plot([1, 2, 3, 4, 5], arr_cummax.mean(0), label=model_name, lw=linewidth, color=colors[i])
        attribute_rewards_cummax = torch.stack(attribute_rewards_cummax)  # Shape: (n_models, n_prompts, n_completions)
        print("attribute_rewards_cummax shape:", attribute_rewards_cummax.shape)
        rewards_cummax.append(attribute_rewards_cummax)
        print(attribute_name)
        plt.legend(loc='lower right')
        plt.ylabel("Reward")
        plt.xlabel("Number of samples: N")
        plt.xticks([1, 2, 3, 4, 5])
        plt.tight_layout()
        plt.savefig(f"figures/bon_{datasplit}_{attribute_name}.pdf")
        plt.show()
    rewards_cummax = torch.stack(rewards_cummax)  # Shape: (n_attribute, n_models, n_prompts, n_completions)
    print("rewards_cummax shape: ", rewards_cummax.shape)
    for i in range(1, rewards_cummax.shape[1]):
        df_winrates = pd.DataFrame(
            torch.mean((rewards_cummax[:, i, :, :] >= rewards_cummax[:, 0, :, :]).float(), 1),
            index=attribute_names,
            columns=list(range(1, 6))
        )
        df_winrates.loc["Average", :] = torch.mean((rewards_cummax[:, i, :, :] >= rewards_cummax[:, 0, :, :]).float(), dim=(0,1)).tolist()
        model_name = list(models_to_compare.keys())[i]
        print(f"\n--- Winrates for {model_name} ---")
        with pd.option_context('display.precision', 3):
            display(df_winrates)

# Validate Non-Transitivity

In [ ]:
reward_paths = list(data_paths.values())
n_completions = []
prompt_ids = None
attribute_names = None
rewards_array = []
for data_path in reward_paths:
    rewards_data = datasets.load_from_disk(data_path)["validation"]
    if prompt_ids is None:
        prompt_ids = rewards_data["prompt_id"]
    else:
        assert all([x == y for x,y in zip(rewards_data["prompt_id"], prompt_ids)])
    rewards_data = rewards_data.remove_columns("prompt_id")
    if attribute_names is None:
        attribute_names = rewards_data.column_names
    arr = [torch.tensor(rewards_data[colname]) for colname in rewards_data.column_names]
    arr = torch.stack(arr)[...,:5]  # Shape: (n_attributes, n_prompts, n_completions)
    n_completions.append(arr.shape[-1])
    rewards_array.append(arr)
rewards_array = torch.concat(rewards_array, -1)
rewards_array.shape

In [ ]:
arr = rewards_array
arr_prob = torch.sigmoid(arr.unsqueeze(-1) - arr.unsqueeze(-2))  # Shape: (attributes, n_prompts, n_completions, n_completions)
arr_prob_mean = torch.mean(arr_prob, 0)  # Shape: (n_prompts, n_completions, n_completions)
print(arr_prob_mean.shape)
arr_prob_mean[0]

In [ ]:
arr_prob_selected = arr_prob[[0,1,2,3,4]]
arr_prob_selected_mean = arr_prob_selected.mean(0)
deterministic_comparisons = torch.mean((arr_prob_selected<0.5).to(dtype=torch.float32), 0)  # Shape: n_prompts, n_completions, n_completions
n_acyclic = 0
n_condorcet = 0
condorcet_indices = []
for i in range(deterministic_comparisons.shape[0]):
    # adj_matrix = (arr_prob_selected_mean[i] > 0.5).numpy().astype(int)
    adj_matrix = (deterministic_comparisons[i] >= 0.5).numpy().astype(int)
    adj_matrix[deterministic_comparisons[i] == 0.5] = 0
    G = nx.DiGraph(adj_matrix)
    is_acyclic = nx.is_directed_acyclic_graph(G)
    condorcet_winner = np.any(np.sum(adj_matrix, 0) == adj_matrix.shape[0]-1)
    condorcet_idx = np.where(np.sum(adj_matrix, 0) == adj_matrix.shape[0]-1)[0]
    if i < 10:
        plt.clf()
        print("Sample: ", i)
        print("Acyclic? ", is_acyclic)
        print("Condorcet winner?", condorcet_winner)
        print("Condorcet idx: ", condorcet_idx)
        models = ["Qwen2.5-0.5B", "RLOO", "Nash-MD-PG", "StackelbergGDA"]
        node_colors = [colors[i] for i in range(4) for _ in range(5)]
        legend_handles = [
            Patch(color=colors[i], label=models[i])
            for i in range(4)
        ]
        
        sink_nodes = [n for n in G.nodes() if G.in_degree(n) > 0 and G.out_degree(n) == 0]
        if len(sink_nodes) > 0:
            node_edgecolors = [colors[len(models)+1] if n in sink_nodes else 'none' for n in G.nodes()]
            legend_handles.append(Patch(facecolor='white', edgecolor=colors[len(models)+1], linewidth=2, label='Condorcet Winner'))

        plt.figure(figsize=(6,6))
        pos = nx.spring_layout(G, k=1.5, seed=42)  # Position nodes
        nx.draw(
            G,
            pos,
            with_labels=True,
            node_color=node_colors,
            edge_color='gray',
            font_color="white",
            node_size=1000,
            font_size=10,
            arrows=True,
            arrowsize=10,
            width=1,
            edgecolors=node_edgecolors,
            linewidths=3
        )
        plt.legend(handles=legend_handles, title="Algorithms", loc='upper left', bbox_to_anchor=(1, 1))
        plt.savefig(f"figures/preference_graph_prompt_idx_{i}.pdf", bbox_inches='tight', dpi=300)
        plt.show()
    if is_acyclic:
        n_acyclic += 1
    if condorcet_winner:
        n_condorcet += 1
        condorcet_indices.extend(condorcet_idx)

print("Number of cyclic prompts: {:.2f}".format((deterministic_comparisons.shape[0] - n_acyclic)/deterministic_comparisons.shape[0]))
print("Number of non-condorcet prompts: {:.2f}".format((deterministic_comparisons.shape[0] - n_condorcet)/deterministic_comparisons.shape[0]))

## Correction capabilities
Use the `correction_evaluation.sh` script to generate the responses and rewards for these metrics.

In [ ]:
models = ["Qwen2.5", "RLOO", "Nash", "Stackelberg"]
data_dir = "../data/experiments/correction_comparison"

comparison_df = {}
for model1, model2 in product(models, models):
    print(model1, model2)
    filename = f"Leader-{model1}__Follower-{model2}__rewards"
    try:
        rewards_data = load_from_disk(os.path.join(data_dir, filename))["validation"]
    except Exception as e:
        print(f"Loading {filename} failed: {e}")
        comparison_df[(model1, model2)] = np.nan
        continue
    
    rewards_data = rewards_data.remove_columns("prompt_id")
    arr = [torch.tensor(rewards_data[colname]) for colname in rewards_data.column_names]
    arr = torch.stack(arr)  # Shape: (n_attributes, n_prompts, 2)
    winrate = torch.mean((arr[...,1] >= arr[...,0]).float()).item()
    comparison_df[(model1, model2)] = winrate
comparison_df = pd.Series(comparison_df).unstack(0).loc[models, models]
with pd.option_context('display.precision', 3):
    display(comparison_df)